In [9]:
# Monta o Google Drive para importar os arquivos
from google.colab import drive
drive.mount('/gdrive')

Drive already mounted at /gdrive; to attempt to forcibly remount, call drive.mount("/gdrive", force_remount=True).


In [0]:
# Importa todos os arquivos necessários (atualize para o caminho correto)
!cat '/gdrive/My Drive/IC/QSARModeling 0.2/Jupyter Notebook/Notebooks/modules/ga.py'
!cat '/gdrive/My Drive/IC/QSARModeling 0.2/Jupyter Notebook/Notebooks/modules/cross_validation_class.py'
!cat '/gdrive/My Drive/IC/QSARModeling 0.2/Jupyter Notebook/Notebooks/modules/yrandomization.py'
!cat '/gdrive/My Drive/IC/QSARModeling 0.2/Jupyter Notebook/Notebooks/modules/lno.py'
!cat '/gdrive/My Drive/IC/QSARModeling 0.2/Jupyter Notebook/Notebooks/modules/filter.py'
!cat '/gdrive/My Drive/IC/QSARModeling 0.2/Jupyter Notebook/Notebooks/modules/lj_cut.py'
!cat '/gdrive/My Drive/IC/QSARModeling 0.2/Jupyter Notebook/Notebooks/modules/validate_yr_lno.py'

In [0]:
# Informa o novo caminho para importar os módulos
import sys
sys.path.append('/gdrive/My Drive/IC/QSARModeling 0.2/Jupyter Notebook/Notebooks/modules/')

In [24]:
# Só é necessário executar se a célula a seguir não encontrar o deap
!pip install deap

In [0]:
# Importing libraries

import numpy as np
import pandas as pd
from ga import Ga
from cross_validation_class import CrossValidation
from yrandomization import YRandomization
from lno import LNO
from filter import variance_cut,correlation_cut,autocorrelation_cut
import lj_cut as lj
from validate_yr_lno import validate
import os

In [0]:
# Open configuration file in order to look for the matrices and the parameters to run
# GA and cross-validation
dfConf = pd.read_csv("/gdrive/My Drive/IC/heliton/Tetronamidas/QSAR_4D/confGA_tetronamides_4D_Assay1.csv",header=None)
directory = dfConf[1][0]
xFile = dfConf[1][1]
yFile = dfConf[1][2]
var_cut = float(dfConf[1][3])
corr_cut = float(dfConf[1][4])
nLVModel = None if dfConf.isnull()[1][5] else int(dfConf[1][5])
min_size = int(dfConf[1][6])
max_size = int(dfConf[1][7])
size_population = int(dfConf[1][8])
mig_rate = float(dfConf[1][9])
cxpb = float(dfConf[1][10])
mutpb = float(dfConf[1][11])
ngen = int(dfConf[1][12])
yr_crit = float(dfConf[1][13])
lno_crit = float(dfConf[1][14])
out_directory = dfConf[1][15]
out_matrix = dfConf[1][16]
out_cv = dfConf[1][17]
Q2_file = dfConf[1][18]
var_sel_file = dfConf[1][19]
autoscale = dfConf[1][20].upper() == "YES"
df = pd.read_csv(os.path.join(directory,xFile),sep=';',index_col=0)
# Filtering the matrix according to the options in configuration file
dfX = lj.transform(df) if dfConf[1][21].upper() == "YES" else df
#dfX = dfX.drop(["teste_alinha/dez_a.mol2","teste_alinha/dez_e.mol2","teste_alinha/onze_a.mol2"])
print("Dimensions of the original matrix")
print(dfX.shape)
y = pd.read_csv(os.path.join(directory,yFile),sep=';',header=None).values
indVar = variance_cut(dfX.values,var_cut)
dfVar = dfX.loc[:,dfX.columns[indVar]]
print("Dimensions of the matrix after variance cut")
print(dfVar.shape)
indCorr = correlation_cut(dfVar.values,y,corr_cut)
dfCorr = dfVar.loc[:,dfVar.columns[indCorr]]
print("Dimensions of the matrix after correlation cut")
print(dfCorr.shape)
auto_cut = dfConf[1][22]
indAuto = autocorrelation_cut(dfCorr.values,y,auto_cut)
dfRest = dfCorr.iloc[:,indAuto]
print("Dimensions of the matrix after auto correlation cut")
print(dfRest.shape)
dfRest.to_csv(os.path.join(out_directory,"filtered_"+out_matrix),sep=';')
X = dfRest.values
if nLVModel == None:
    nLVModel = dfCorr.shape[0]
ga = Ga(X,y,nLVModel, autoscale, min_size, max_size, size_population, mig_rate, cxpb, mutpb, ngen)
ga.run()
ga.saveQ2(os.path.join(out_directory,Q2_file))
ga.savePop(os.path.join(out_directory,var_sel_file))
Q2 = ga.Q2
Q2 = [Q2[i][0] for i,_ in enumerate(Q2)]
var_sel = validate(X,y,ga.pop_selected,Q2,yr_cut=yr_crit,lno_cut=lno_crit)
if var_sel != []:
    dfSel = dfRest.loc[:,dfRest.columns[var_sel]]
    dfSel.to_csv(os.path.join(out_directory,out_matrix),sep=';')
    cv = CrossValidation(dfSel.values,y)
    cv.saveParameters(os.path.join(out_directory,out_cv))        
else:
    print("y-randomization or LNO failed!")

Caso a execução já tenha sido começada em outro momento e interrompida, a matriz cortada pela autocorrelação pode ser reaproveitada com o código abaixo, poupando trabalho computacional.

In [0]:
 dfConf = pd.read_csv("/gdrive/My Drive/IC/heliton/Tetronamidas/QSAR_4D/confGA_tetronamides_4D_Assay1.csv",header=None)
directory = dfConf[1][0]
xFile = dfConf[1][1]
yFile = dfConf[1][2]
var_cut = float(dfConf[1][3])
corr_cut = float(dfConf[1][4])
nLVModel = None if dfConf.isnull()[1][5] else int(dfConf[1][5])
min_size = int(dfConf[1][6])
max_size = int(dfConf[1][7])
size_population = int(dfConf[1][8])
mig_rate = float(dfConf[1][9])
cxpb = float(dfConf[1][10])
mutpb = float(dfConf[1][11])
ngen = int(dfConf[1][12])
yr_crit = float(dfConf[1][13])
lno_crit = float(dfConf[1][14])
out_directory = dfConf[1][15]
out_matrix = dfConf[1][16]
out_cv = dfConf[1][17]
Q2_file = dfConf[1][18]
var_sel_file = dfConf[1][19]
autoscale = dfConf[1][20].upper() == "YES"
df = pd.read_csv(os.path.join(directory,xFile),sep=';',index_col=0)
# Filtering the matrix according to the options in configuration file
dfX = lj.transform(df) if dfConf[1][21].upper() == "YES" else df
#dfX = dfX.drop(["teste_alinha/dez_a.mol2","teste_alinha/dez_e.mol2","teste_alinha/onze_a.mol2"])
print("Dimensions of the original matrix")
print(dfX.shape)
y = pd.read_csv(os.path.join(directory,yFile),sep=';',header=None).values
indVar = variance_cut(dfX.values,var_cut)
dfVar = dfX.loc[:,dfX.columns[indVar]]
print("Dimensions of the matrix after variance cut")
print(dfVar.shape)
indCorr = correlation_cut(dfVar.values,y,corr_cut)
dfCorr = dfVar.loc[:,dfVar.columns[indCorr]]
print("Dimensions of the matrix after correlation cut")
print(dfCorr.shape)
dfRest = pd.read_csv(os.path.join(out_directory,"filtered_"+out_matrix),sep=';',index_col=0)
print("Dimensions of the matrix after auto correlation cut")
print(dfRest.shape)
X = dfRest.values
if nLVModel == None:
    nLVModel = dfRest.shape[0]
ga = Ga(X,y,nLVModel, autoscale, min_size, max_size, size_population, mig_rate, cxpb, mutpb, ngen)
ga.run()
ga.saveQ2(os.path.join(out_directory,Q2_file))
ga.savePop(os.path.join(out_directory,var_sel_file))
Q2 = ga.Q2
Q2 = [Q2[i][0] for i,_ in enumerate(Q2)]
var_sel = validate(X,y,ga.pop_selected,Q2,yr_cut=yr_crit,lno_cut=lno_crit)
if var_sel != []:
    dfSel = dfRest.loc[:,dfRest.columns[var_sel]]
    dfSel.to_csv(os.path.join(out_directory,out_matrix),sep=';')
    cv = CrossValidation(dfSel.values,y)
    cv.saveParameters(os.path.join(out_directory,out_cv))        
else:
    print("y-randomization or LNO failed!")

In [0]:
var_sel = validate(X,y,ga.pop_selected,Q2,yr_cut=yr_crit,lno_cut=lno_crit)

In [0]:
print(var_sel)

[]


In [0]:
with open(os.path.join(out_directory,'1_Q2_out_.json'),'r') as file:
    Q2 = json.loads(file.read())
with open(os.path.join(out_directory,'1_Pop_out_.json'),'r') as file:
    Pop = json.loads(file.read())
Q2 = [Q2[i][0] for i,_ in enumerate(Q2)]

In [29]:
var_sel = validate(X,y,Pop,Q2,yr_cut=yr_crit,lno_cut=lno_crit)
if var_sel != []:
    dfSel = dfRest.loc[:,dfRest.columns[var_sel]]
    dfSel.to_csv(os.path.join(out_directory,out_matrix),sep=';')
    cv = CrossValidation(dfSel.values,y)
    cv.saveParameters(os.path.join(out_directory,out_cv))        

ValueError: ignored